# Day 47: Multi-class & Multi-label Classification

## Multi-class Recap
"Multi-class" means each sample gets exactly one label, chosen from three or more possible classes (as opposed to binary classification, which only picks between two). Most scikit-learn classifiers, including `LogisticRegression`, support this natively without any extra setup - internally they either generalize the algorithm's math directly to multiple classes (e.g. softmax regression) or automatically train multiple binary classifiers behind the scenes and combine their outputs. This is important context before we get to the OvR and multi-label sections below, which show what's actually happening under the hood and how multi-label (multiple simultaneous labels) differs from multi-class (one label out of many). The code below trains a logistic regression on the Iris dataset (3 species) and just confirms which classes the model learned to distinguish.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LogisticRegression(max_iter=200).fit(X_train, y_train)  # max_iter raised so the solver fully converges
print('Classes:', model.classes_)  # the 3 species labels the model learned to distinguish

## One-vs-Rest Strategy
Some algorithms (like the base `SVC`) don't naturally extend to more than two classes, so scikit-learn provides wrapper strategies to adapt them. One-vs-Rest (OvR) is the most common: it trains one separate binary classifier per class, where each classifier learns to distinguish "is this class" vs. "is anything else." At prediction time, all classifiers score the sample, and OvR picks the class whose classifier is most confident. This is simple, easy to parallelize, and works with any binary classifier as the base estimator, though it can struggle a bit when classes overlap heavily since each binary classifier is solving a slightly different, imbalanced sub-problem (its class vs. everyone else combined). Here we wrap an `SVC` (which uses OvR/OvO internally by default anyway, but this makes the strategy explicit) and check its accuracy on the same Iris test set as before, so you can compare it to the plain multi-class logistic regression.

In [ ]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC

ovr_model = OneVsRestClassifier(SVC()).fit(X_train, y_train)  # trains one binary SVC per class ("this class" vs "rest")
print('OvR accuracy:', ovr_model.score(X_test, y_test))

## Multi-label Classification
Multi-label is a different problem from multi-class: instead of picking exactly one label out of several, each sample can independently be assigned zero, one, or several labels at once - a movie can be tagged both 'Comedy' and 'Romance' simultaneously, an email could be both 'Work' and 'Urgent'. This means the target `y` is no longer a single column of class labels but a 2D array/matrix, one binary column per possible label indicating whether that label applies. A common way to handle this is `MultiOutputClassifier`, which under the hood trains one independent binary classifier per label column (conceptually similar to OvR, but for labels that aren't mutually exclusive rather than for classes that are). Here `make_multilabel_classification` generates a synthetic dataset with 4 possible labels where samples can have multiple labels active, and the printed prediction for the first 3 samples will show a row of 0s/1s per sample - a 1 in position i means label i is predicted to apply, and multiple 1s can appear in the same row (unlike multi-class, where only one class can be "on").

In [ ]:
from sklearn.datasets import make_multilabel_classification
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier

X_ml, y_ml = make_multilabel_classification(n_samples=200, n_classes=4, n_labels=2, random_state=42)  # y_ml is a 200x4 binary matrix, ~2 labels active per sample on average
model_ml = MultiOutputClassifier(RandomForestClassifier(random_state=42)).fit(X_ml, y_ml)  # trains one classifier per label column
print('Sample prediction (multiple labels):', model_ml.predict(X_ml[:3]))  # each row can have more than one 1 (multiple labels active)